<a href="https://colab.research.google.com/github/Sophieli0113/ds2002-fa26/blob/main/notebooks/08-ETL%20and%20the%20Walmart%20case/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PATH = '/content/drive/MyDrive/ds2002/product_catalog.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
# PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [3]:
# TODO (hint: .str.lower().str.contains('pop'))
mask = cat['product_name'].str.lower().str.contains('pop', na=False)
pop = cat[mask]

print(len(pop))
pop

4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [4]:
# TODO
pt = cat[cat['product_name'].str.contains('pop', case=False, na=False)]

is_straw = pt['product_name'].str.contains('strawberry', case=False, na=False)
straw = pt[is_straw]
other_pop = pt[~is_straw]

print("Strawberry Pop-Tart SKUs and names:")
print(straw[['sku', 'product_name']].to_string(index=False))

print(f"\nNon-strawberry Pop-Tart rows: {len(other_pop)}. None present; any other flavor would stay separate.")
other_pop

Strawberry Pop-Tart SKUs and names:
          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry
PT_STRAWBERRY pop-tarts strawberry

Non-strawberry Pop-Tart rows: 0. None present; any other flavor would stay separate.


,sku,product_name,category,subcategory,unit_cost,supplier,package_size


**Pop-Tart rows that are a different product:** none are present.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [5]:
# TODO
sku_map = {
    'SKU#459812':    'PT-12',
    'POPTART-STRAW': 'PT-12',
    'PT_STRAWBERRY': 'PT-12',
}

cat['sku_canonical'] = cat['sku'].replace(sku_map)

# check
print(cat['sku_canonical'].isna().sum())
cat[['sku', 'sku_canonical', 'product_name']].head(6)

0


,sku,sku_canonical,product_name
0,PT-12,PT-12,pop-tarts strawberry
1,SKU#459812,PT-12,Strawberry Pop-Tart
2,POPTART-STRAW,PT-12,Pop-Tarts Strawberry
3,PT_STRAWBERRY,PT-12,pop-tarts strawberry
4,WTR-24,WTR-24,BOTTLED WATER 24PK
5,WATER24PK,WATER24PK,Bottled Water 24-Pack


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [6]:
# TODO
print("Before:")
print(cat['category'].value_counts(dropna=False))

# merge pairs to same categories
cat_map = {
    'Emergency Supplies':  'Emergency',
    'Groceries':           'Grocery',
    'Breakfast Foods':     'Breakfast',
    'Baby Products':       'Baby',
    'Alcoholic Beverages': 'Alcohol',
}

cat['category_clean'] = (
    cat['category'].str.strip().str.title().replace(cat_map)
)

print("\nAfter:")
print(cat['category_clean'].value_counts(dropna=False))

Before:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

After:
category_clean
Emergency        12
Grocery           6
Entertainment     5
Breakfast         4
Electronics       4
Baby              4
Household         4
Alcohol           2
Snacks            2
Health            2
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
# TODO
cat['normalized'] = (
    cat['product_name'].str.lower()
    .str.replace(r'[^a-z ]', '', regex=True)
    .str.split()
    .apply(lambda w: ' '.join(sorted(w)))
)

candidates = (
    cat.groupby('normalized')['sku']
    .agg(list)
    .loc[lambda s: s.str.len() > 1]
)
candidates

,sku
normalized,
aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
beer light pack,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"
chips pack variety,"[CHIPS-VP, CHIPVARIETY]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
# TODO
n_before = len(cat)

sku_map = {
    'SKU#459812':    'PT-12',
    'POPTART-STRAW': 'PT-12',
    'PT_STRAWBERRY': 'PT-12',
}
cat['sku_canonical'] = cat['sku'].replace(sku_map)

assert cat['sku_canonical'].isna().sum() == 0, "mapping produced nulls"
assert len(cat) == n_before, f"row count changed: {n_before} -> {len(cat)}"
print(f"OK: {len(cat)} rows, 0 nulls in sku_canonical")

OK: 45 rows, 0 nulls in sku_canonical


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

(a) Monopoly is split across three SKUs (BG-MONO, MONOPOLY-01, and GAME-MONO), and the normalized grouping picked those up pretty easily. I’d just map them all to one SKU, though I’d double-check first that they’re actually the same edition. Bottled water was more interesting, since WTR-24 and WATER24PK are obviously the same 24-pack, but the grouping didn’t catch them because “pk” and “pack” end up as different words. For that one I’d either add the pair by hand or change “pk” to “pack” before grouping, and I’d be careful not to accidentally merge it with a different pack size.

(b) It becomes a business decision when you’re deciding whether two actually different items should count as one. I’m not sure about POPTART-STRAW, which is a 6-pack while the other strawberry ones are 4-packs. Merging them makes sense for seeing overall demand, but for restocking it might matter which size people are buying. I’d probably ask the operations team, since they’re the ones deciding what goes on the trucks.